# 07 · Spark SQL

Everything you've done with the DataFrame API you can also do in plain **SQL** — and you can mix the two
freely. Both compile to the *same* optimized plan, so there's no performance difference. Use whichever
reads better for the task (many teams write transformations in SQL).

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA}/products.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)

# register DataFrames as temporary views so SQL can see them
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
customers.createOrReplaceTempView("customers")

In [ ]:
spark.sql("SELECT status, COUNT(*) AS n FROM orders GROUP BY status ORDER BY n DESC").show()

Multi-line SQL with joins and a CTE (`WITH`):

In [ ]:
result = spark.sql("""
    WITH completed AS (
        SELECT o.*, p.category, o.quantity * p.price AS revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'completed'
    )
    SELECT c.country, ROUND(SUM(revenue), 2) AS revenue, COUNT(*) AS n_orders
    FROM completed
    JOIN customers c USING (customer_id)
    GROUP BY c.country
    ORDER BY revenue DESC
""")
result.show()

`spark.sql` returns an ordinary DataFrame, so you can keep going with the Python API:

In [ ]:
result.filter(F.col("country").isNotNull()).withColumn("avg_order", F.round(F.col("revenue") / F.col("n_orders"), 2)).show(3)

### Same plan either way

In [ ]:
spark.sql("SELECT customer_id, COUNT(*) AS n FROM orders WHERE quantity > 2 GROUP BY customer_id").explain()
orders.filter(F.col("quantity") > 2).groupBy("customer_id").agg(F.count("*").alias("n")).explain()

## SQL snippets inside the DataFrame API
* `F.expr("...")` — any SQL expression as a Column
* `df.selectExpr("...", "...")` — select with SQL expressions
* `df.filter("...")` — filter with a SQL condition

In [ ]:
orders.selectExpr("order_id", "quantity * 2 AS double_qty", "upper(status) AS status").show(3)
orders.withColumn("is_big", F.expr("CASE WHEN quantity >= 4 THEN true ELSE false END")).show(3)
orders.filter("status = 'returned' AND quantity > 3").show(3)

## Parameterised queries
Never paste user input into SQL with f-strings (SQL injection!). Use named parameters:

In [ ]:
spark.sql("SELECT * FROM orders WHERE status = :status AND quantity >= :q",
          args={"status": "returned", "q": 5}).show(3)

## Views and the catalog
* `createOrReplaceTempView` — visible to this SparkSession only, disappears when it stops.
* `createOrReplaceGlobalTempView` — shared across sessions in the same app, lives in the `global_temp` database.
* `df.write.saveAsTable("name")` — a permanent table in a metastore (on Databricks this is how you create real tables).

In [ ]:
for t in spark.catalog.listTables():
    print(t.name, t.tableType)

### Exercise 1
Using **`spark.sql`**, create `sql_rev_by_cat`: completed-order revenue per category, columns **`category`, `revenue`** (rounded to 2 dp).

In [ ]:
sql_rev_by_cat = None  # TODO  spark.sql("""...""")

In [ ]:
# Check — run me after your answer
got = {r.category: r.revenue for r in sql_rev_by_cat.collect()}
exp = {'Books': 346914.81, 'Clothing': 174227.66, 'Electronics': 270553.15, 'Home': 234515.31, 'Toys': 374985.12}
assert got.keys() == exp.keys() and all(abs(got[k] - exp[k]) < 0.05 for k in exp), got
print("Correct!")

### Exercise 2
Using **SQL with a window function** (`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`), create `sql_top2`: the **top 2 products by completed revenue in each category**. Columns **`category`, `product_name`, `revenue`, `rnk`**.

In [ ]:
sql_top2 = None  # TODO

In [ ]:
# Check — run me after your answer
assert sql_top2.columns == ["category", "product_name", "revenue", "rnk"]
assert sorted(r.product_name for r in sql_top2.collect()) == ['Cookbook', 'Jacket', 'Jeans', 'Lamp', 'Monitor', 'Pillow', 'Plush Bear', 'Poetry Book', 'Puzzle', 'Speaker']
print("Correct!")

### Exercise 3
Using **`selectExpr`** on `orders`, create `sx` with columns **`order_id`** and **`qty_x10`** (quantity × 10).

In [ ]:
sx = None  # TODO

In [ ]:
# Check — run me after your answer
assert sx.columns == ["order_id", "qty_x10"]
assert sx.agg(F.sum("qty_x10")).first()[0] == orders.agg(F.sum("quantity")).first()[0] * 10
print("Correct!")

---
Stuck? Worked answers (with outputs) are in [`../solutions/07_spark_sql.ipynb`](../solutions/07_spark_sql.ipynb).